# The `mediapipe` box — hands, pose and the face mesh

Every feature of the box, one section each, against a running fleet.

| command | what comes back | parameters that matter |
|---|---|---|
| `hands` *(default)* | 21 landmarks per hand + `world_landmarks`, `handedness`, `scores` | `max_results` |
| `pose` | 33 landmarks per person + `world_landmarks`, `visibility` | `model` (`lite`/`full`/`heavy`) |
| `face` | the 478-point mesh | `blendshapes`, `transformation_matrix` |
| `reset` | closes the cached landmarkers | — |

Two things this notebook spends time on, because they are what callers get wrong:

* **`landmarks` is padded** to the largest detection count in the *batch* — `counts`
  is the only honest way to know how many rows are real (section 3);
* **coordinates are normalized `(x, y, z)`**, not pixels, and not the `(y, x)` that
  `tapnext` reports (section 4).

**You need:** the box running — `cd fleet && docker compose up -d mediapipe`
(host port **9076**) — and internet on first run, to cache three sample photos.
Authoritative request shape:
[`boxes/mediapipe/README.md`](https://github.com/sipg-isr/VisionIST_Library/blob/main/boxes/mediapipe/README.md).

In [ ]:
%pip install -q visionist_client numpy matplotlib pillow

## 0. Settings

In [ ]:
import os, pathlib

# The box. Override with MEDIAPIPE_HOST=sipgpu1.isrnet:9076 for a remote fleet.
MEDIAPIPE_HOST = os.environ.get("MEDIAPIPE_HOST", "localhost:9076")

# Sample photos are cached here on first run. Point these at your own files to
# use your own images - any JPEG/PNG path works.
SAMPLES = pathlib.Path("samples"); SAMPLES.mkdir(exist_ok=True)

SAMPLE_URLS = {
    # the images the upstream MediaPipe task notebooks use
    "hands": "https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg",
    "pose":  "https://cdn.pixabay.com/photo/2019/03/12/20/39/girl-4051811_960_720.jpg",
    "face":  "https://storage.googleapis.com/mediapipe-assets/business-person.png",
}

## 1. Helpers

One cell of plumbing: a cached download, the call wrapper, and the drawing code.
The skeleton edge lists are MediaPipe's own, inlined — so the notebook needs no
`mediapipe` install on this side, only the box.

In [ ]:
import time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from PIL import Image
from visionist_client import Visionist

mp = Visionist(MEDIAPIPE_HOST, config_key="mediapipe")


def sample(name):
    """Local path of a sample image, downloaded once."""
    url = SAMPLE_URLS[name]
    path = SAMPLES / f"{name}{pathlib.Path(url).suffix}"
    if not path.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "visionist-notebook"})
        with urllib.request.urlopen(req, timeout=60) as r:
            path.write_bytes(r.read())
    return path


def rgb(path):
    """The image as an (H, W, 3) uint8 array, for plotting."""
    return np.asarray(Image.open(path).convert("RGB"))


def mediapipe(command, images=(), **parameters):
    """One call. Returns (Result, the box's reply section).

    Raises on an error reply, so a cell stops at the box's own message instead
    of failing later on a missing field.
    """
    data = {"images": [pathlib.Path(p) for p in images]} if images else {}
    res = mp.run(data=data,
                 config={"mediapipe": {"command": command, "parameters": parameters}})
    section = (res.config or {}).get("mediapipe", {})
    if section.get("status") == "error":
        raise RuntimeError(f"mediapipe box: {section.get('error')}")
    return res, section


def found(res, i=0):
    """Rows of `landmarks` that are real for image i - the counts contract."""
    n = int(np.asarray(res.counts).ravel()[i])
    return np.asarray(res.landmarks)[i, :n]


# ---- drawing ---------------------------------------------------------------
# Three categorical hues, fixed order, never cycled. Marks get a white ring so
# they stay legible over any photo.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
RING = "white"


def skeleton(ax, pts, edges, color, point_size=14, label=None):
    """Draw one detection: edges then points, each over a white relief."""
    for a, b in edges:
        xs, ys = [pts[a, 0], pts[b, 0]], [pts[a, 1], pts[b, 1]]
        ax.plot(xs, ys, color=RING, lw=4, solid_capstyle="round", zorder=2)
        ax.plot(xs, ys, color=color, lw=2, solid_capstyle="round", zorder=3)
    ax.scatter(pts[:, 0], pts[:, 1], s=point_size, color=color,
               edgecolors=RING, linewidths=1.2, zorder=4)
    if label:                     # identity is never colour alone
        ax.annotate(label, (pts[0, 0], pts[0, 1]), xytext=(6, -14),
                    textcoords="offset points", color=color, fontsize=11,
                    fontweight="bold", zorder=5,
                    path_effects=[pe.withStroke(linewidth=3, foreground=RING)])


def canvas(image, title, figsize=(8, 8)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image); ax.set_axis_off(); ax.set_title(title, fontsize=11)
    return fig, ax


HAND_EDGES = [
    (0, 1), (0, 5), (9, 13), (13, 17), (5, 9), (0, 17), (1, 2), (2, 3), (3, 4), (5, 6),
    (6, 7), (7, 8), (9, 10), (10, 11), (11, 12), (13, 14), (14, 15), (15, 16),
    (17, 18), (18, 19), (19, 20)
]

POSE_EDGES = [
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), (9, 10), (11, 12),
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19), (12, 14), (14, 16),
    (16, 18), (16, 20), (16, 22), (18, 20), (11, 23), (12, 24), (23, 24), (23, 25),
    (24, 26), (25, 27), (26, 28), (27, 29), (28, 30), (29, 31), (30, 32), (27, 31),
    (28, 32)
]

FACE_CONTOURS = [
    (61, 146), (146, 91), (91, 181), (181, 84), (84, 17), (17, 314), (314, 405),
    (405, 321), (321, 375), (375, 291), (61, 185), (185, 40), (40, 39), (39, 37),
    (37, 0), (0, 267), (267, 269), (269, 270), (270, 409), (409, 291), (78, 95),
    (95, 88), (88, 178), (178, 87), (87, 14), (14, 317), (317, 402), (402, 318),
    (318, 324), (324, 308), (78, 191), (191, 80), (80, 81), (81, 82), (82, 13),
    (13, 312), (312, 311), (311, 310), (310, 415), (415, 308), (263, 249), (249, 390),
    (390, 373), (373, 374), (374, 380), (380, 381), (381, 382), (382, 362), (263, 466),
    (466, 388), (388, 387), (387, 386), (386, 385), (385, 384), (384, 398), (398, 362),
    (276, 283), (283, 282), (282, 295), (295, 285), (300, 293), (293, 334), (334, 296),
    (296, 336), (33, 7), (7, 163), (163, 144), (144, 145), (145, 153), (153, 154),
    (154, 155), (155, 133), (33, 246), (246, 161), (161, 160), (160, 159), (159, 158),
    (158, 157), (157, 173), (173, 133), (46, 53), (53, 52), (52, 65), (65, 55),
    (70, 63), (63, 105), (105, 66), (66, 107), (10, 338), (338, 297), (297, 332),
    (332, 284), (284, 251), (251, 389), (389, 356), (356, 454), (454, 323), (323, 361),
    (361, 288), (288, 397), (397, 365), (365, 379), (379, 378), (378, 400), (400, 377),
    (377, 152), (152, 148), (148, 176), (176, 149), (149, 150), (150, 136), (136, 172),
    (172, 58), (58, 132), (132, 93), (93, 234), (234, 127), (127, 162), (162, 21),
    (21, 54), (54, 103), (103, 67), (67, 109), (109, 10)
]

FACE_IRISES = [
    (474, 475), (475, 476), (476, 477), (477, 474), (469, 470), (470, 471), (471, 472),
    (472, 469)
]

POSE_NAMES = [
    "nose", "left_eye_inner", "left_eye", "left_eye_outer",
    "right_eye_inner", "right_eye", "right_eye_outer", "left_ear",
    "right_ear", "mouth_left", "mouth_right", "left_shoulder",
    "right_shoulder", "left_elbow", "right_elbow", "left_wrist",
    "right_wrist", "left_pinky", "right_pinky", "left_index",
    "right_index", "left_thumb", "right_thumb", "left_hip",
    "right_hip", "left_knee", "right_knee", "left_ankle",
    "right_ankle", "left_heel", "right_heel", "left_foot_index",
    "right_foot_index",
]

In [ ]:
# Is it up? `info()` uses gRPC reflection, no model work.
print(mp.info())

res, section = mediapipe("reset")
print(section)          # {'status': 'done', 'action': 'reset'}

## 2. `hands` — 21 landmarks, handedness, and metric world landmarks

The default command. `max_results` is the upstream `num_hands`; the box defaults
to 2 for hands (1 for pose and face).

In [ ]:
img_path = sample("hands")
img = rgb(img_path)
print("image", img.shape[1], "x", img.shape[0])

res, section = mediapipe("hands", [img_path], max_results=2, coordinates="pixels")
section

In [ ]:
for name, value in res.fields.items():
    print(f"{name:18s} {np.asarray(value).shape}  {np.asarray(value).dtype}")

print("\ncounts      :", np.asarray(res.counts).ravel())
print("handedness  :", np.asarray(res.handedness))   # 0 left · 1 right · -1 padding
print("scores      :", np.round(np.asarray(res.scores), 3))

`handedness` is the label **as seen in the image** — a webcam frame that was not
un-mirrored reports the opposite hand. The `-1` entries, if any, are padding.

In [ ]:
hands = found(res)                       # (n_hands, 21, 3) in pixels
codes = np.asarray(res.handedness)[0]
scores = np.asarray(res.scores)[0]

fig, ax = canvas(img, f"hands · {len(hands)} found · pixel coordinates")
for j, pts in enumerate(hands):
    side = "left" if codes[j] == 0 else "right"
    skeleton(ax, pts, HAND_EDGES, BLUE if codes[j] == 0 else ORANGE,
             label=f"{side} {scores[j]:.2f}")
plt.show()

### `world_landmarks` — the same hand in metres

Normalized landmarks say *where in the frame*; world landmarks say *what shape*,
in metres, with the origin at the hand's geometric centre. They do not move when
the hand moves across the frame, which is what makes them the ones to measure
with.

In [ ]:
world = np.asarray(res.world_landmarks)[0, 0]     # (21, 3), metres

fig = plt.figure(figsize=(5.5, 5.5))
ax = fig.add_subplot(projection="3d")
for a, b in HAND_EDGES:
    ax.plot(*zip(world[a], world[b]), color=BLUE, lw=2)
ax.scatter(*world.T, s=14, color=BLUE, edgecolors=RING, linewidths=0.8, depthshade=False)
ax.set_title("hand 0 · world_landmarks (metres)", fontsize=11)
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.set_box_aspect([np.ptp(world[:, i]) or 1 for i in range(3)])
plt.show()

span = np.ptp(world, axis=0)
print(f"bounding box: {span[0]*100:.1f} x {span[1]*100:.1f} x {span[2]*100:.1f} cm")

## 3. The padding contract — always read `counts`

`landmarks` is `(n_images, M, L, 3)` where **M is the largest detection count in
the batch**, so a frame with fewer detections is zero-padded. Zero is a legal
landmark position, so there is no sentinel to test for: `counts[i]` is the answer.

Send one image with hands and one with nothing in it to see both halves.

In [ ]:
# a picture with no person in it, made locally
blank = SAMPLES / "no_people.png"
if not blank.exists():
    g = np.linspace(40, 200, 320, dtype=np.uint8)
    Image.fromarray(np.dstack([np.tile(g, (240, 1))] * 3)).save(blank)

res_b, section_b = mediapipe("hands", [img_path, blank], max_results=2)
print("counts         :", np.asarray(res_b.counts).ravel())
print("landmarks shape:", np.asarray(res_b.landmarks).shape)
print("num_detections :", section_b["num_detections"])

for i in range(2):
    rows = found(res_b, i)
    print(f"  image {i}: {len(rows)} real row(s); "
          f"padding rows all zero: {np.all(np.asarray(res_b.landmarks)[i, len(rows):] == 0)}")

In [ ]:
# Nothing anywhere in the batch -> M collapses to 0, not a fabricated row.
res_e, section_e = mediapipe("hands", [blank])
print(np.asarray(res_e.landmarks).shape, np.asarray(res_e.counts).ravel(),
      section_e["status"])

## 4. Coordinates — normalized by default, `pixels` on request

MediaPipe puts `x` and `y` in `[0, 1]` across the width and height, and `z` on
roughly the same scale as `x` (depth relative to the subject, negative towards
the camera). `coordinates="pixels"` multiplies x and z by the width and y by the
height — which is what lines these up with the `opencv` and `features` keypoints.

Worth saying once: this is `(x, y, z)`. `tapnext` reports `(y, x)`.

In [ ]:
res_n, _ = mediapipe("hands", [img_path], coordinates="normalized")
res_p, _ = mediapipe("hands", [img_path], coordinates="pixels")

h, w = img.shape[:2]
norm, pix = found(res_n)[0], found(res_p)[0]
print("normalized [0:3]:\n", np.round(norm[:3], 4))
print("pixels     [0:3]:\n", np.round(pix[:3], 1))
print("\npixels == normalized * (w, h, w):",
      np.allclose(pix, norm * [w, h, w], atol=1e-3))

## 5. `pose` — 33 landmarks, three bundles, per-joint visibility

In [ ]:
pose_path = sample("pose")
pose_img = rgb(pose_path)

timings = {}
for bundle in ("lite", "full", "heavy"):
    t0 = time.perf_counter()
    r, s = mediapipe("pose", [pose_path], model=bundle, coordinates="pixels")
    timings[bundle] = (time.perf_counter() - t0, s["runtime"], found(r)[0])
    print(f"{bundle:6s} wall {timings[bundle][0]:5.2f}s  box {s['runtime']:5.2f}s  "
          f"counts {np.asarray(r.counts).ravel()}")

ref = timings["heavy"][2]
for bundle in ("lite", "full"):
    d = np.linalg.norm(timings[bundle][2][:, :2] - ref[:, :2], axis=1)
    print(f"{bundle:6s} vs heavy: median {np.median(d):5.1f} px, max {d.max():5.1f} px")

The first call to a bundle builds the landmarker (tens of ms) and the box keeps
it; changing `model`, `max_results` or a threshold rebuilds it, because those are
baked into the MediaPipe graph rather than passed per call.

### `visibility` — an occluded joint still gets a position

Every landmark comes back with coordinates whether or not it was seen. `visibility`
in `[0, 1]` is how much to trust each one; here it sizes and shades the points,
with the low-confidence joints named in the margin.

In [ ]:
res_pose, section_pose = mediapipe("pose", [pose_path], model="full", coordinates="pixels")
pts = found(res_pose)[0]                       # (33, 3) pixels
vis = np.asarray(res_pose.visibility)[0, 0]    # (33,)

fig, ax = canvas(pose_img, "pose · point shade and size = visibility", figsize=(7, 9))
for a, b in POSE_EDGES:
    xs, ys = [pts[a, 0], pts[b, 0]], [pts[a, 1], pts[b, 1]]
    ax.plot(xs, ys, color=RING, lw=4, solid_capstyle="round", zorder=2)
    ax.plot(xs, ys, color=BLUE, lw=2, solid_capstyle="round", zorder=3)
sc = ax.scatter(pts[:, 0], pts[:, 1], c=vis, cmap="Blues", vmin=0, vmax=1,
                s=20 + 90 * vis, edgecolors=RING, linewidths=1.2, zorder=4)
fig.colorbar(sc, ax=ax, shrink=0.5, label="visibility")
plt.show()

order = np.argsort(vis)
print("least visible joints:")
for k in order[:5]:
    print(f"  {POSE_NAMES[k]:18s} {vis[k]:.3f}")

In [ ]:
# World landmarks again: metres, origin at the hip midpoint. Scale-free, so this
# is the one to measure limb lengths or joint angles with.
pw = np.asarray(res_pose.world_landmarks)[0, 0]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
for a, b in POSE_EDGES:
    ax.plot(*zip(pw[a], pw[b]), color=BLUE, lw=2)
ax.scatter(*pw.T, s=12, color=BLUE, edgecolors=RING, linewidths=0.8, depthshade=False)
ax.view_init(elev=-80, azim=-90)      # y grows downwards in image convention
ax.set_title("pose · world_landmarks (metres, origin at the hips)", fontsize=11)
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
plt.show()

def seg(a, b):
    return np.linalg.norm(pw[POSE_NAMES.index(a)] - pw[POSE_NAMES.index(b)])

print(f"shoulder width      {seg('left_shoulder', 'right_shoulder')*100:5.1f} cm")
print(f"left upper arm      {seg('left_shoulder', 'left_elbow')*100:5.1f} cm")
print(f"left forearm        {seg('left_elbow', 'left_wrist')*100:5.1f} cm")

## 6. `face` — the 478-point mesh, blendshapes, and the 4×4 matrix

The two extras are off by default: they cost nothing to ask for but change the
reply's field set, which is why they are parameters rather than always-on.

In [ ]:
face_path = sample("face")
face_img = rgb(face_path)

res_face, section_face = mediapipe("face", [face_path], coordinates="pixels",
                                   blendshapes=True, transformation_matrix=True)
print({k: np.asarray(v).shape for k, v in res_face.fields.items()
       if k != "blendshape_names"})
print("names:", len(res_face.blendshape_names), "e.g.", res_face.blendshape_names[:4])
section_face

In [ ]:
mesh = found(res_face)[0]              # (478, 3) pixels

fig, ax = canvas(face_img, "face · 478-point mesh, contours and irises")
for a, b in FACE_CONTOURS:
    ax.plot([mesh[a, 0], mesh[b, 0]], [mesh[a, 1], mesh[b, 1]],
            color=BLUE, lw=1.2, alpha=0.9, zorder=3)
for a, b in FACE_IRISES:
    ax.plot([mesh[a, 0], mesh[b, 0]], [mesh[a, 1], mesh[b, 1]],
            color=ORANGE, lw=1.6, zorder=4)
ax.scatter(mesh[:, 0], mesh[:, 1], s=1.5, color=AQUA, alpha=0.8, zorder=2)
plt.show()

### Blendshapes — 52 expression scores

`blendshape_names` carries the column order, so the scores are never read by
index alone.

In [ ]:
scores = np.asarray(res_face.blendshapes)[0, 0]      # (52,)
names = res_face.blendshape_names
top = np.argsort(scores)[::-1][:12][::-1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([names[i] for i in top], scores[top], color=BLUE, height=0.7)
for y, i in enumerate(top):
    ax.annotate(f"{scores[i]:.2f}", (scores[i], y), xytext=(4, 0),
                textcoords="offset points", va="center", fontsize=9, color="#52514e")
ax.set_xlim(0, max(float(scores[top].max()) * 1.18, 0.05))
ax.set_xlabel("score"); ax.set_title("face · strongest blendshapes", fontsize=11)
ax.xaxis.grid(True, color="#e6e5e1"); ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# The 4x4 maps the canonical face model onto this face: rotation + translation,
# which is the face bundle's answer to "where is the head in space" (it has no
# world_landmarks).
T = np.asarray(res_face.transformation_matrixes)[0, 0]
np.set_printoptions(precision=3, suppress=True)
print(T)
print("\ntranslation (model units):", T[:3, 3])
print("rotation is orthonormal   :", np.allclose(T[:3, :3] @ T[:3, :3].T, np.eye(3), atol=1e-3))

## 7. The rest of the surface: `reset`, and what the errors look like

Every failure comes back as `status: "error"` with a sentence, never an exception
on the wire. The helper raises, so each line below is caught to show the message.

In [ ]:
for label, call in [
    ("unknown command",        lambda: mediapipe("holistic", [img_path])),
    ("bad pose bundle",        lambda: mediapipe("pose", [img_path], model="enormous")),
    ("confidence out of range", lambda: mediapipe("hands", [img_path],
                                                  min_detection_confidence=2)),
    ("not an image",           lambda: mp.run(data={"images": [b"not an image"]},
                                              config={"mediapipe": {"command": "hands"}})),
]:
    try:
        out = call()
        cfg = out[1] if isinstance(out, tuple) else (out.config or {}).get("mediapipe", {})
        print(f"{label:24s} -> {cfg.get('status')}: {cfg.get('error', '')}")
    except RuntimeError as e:
        print(f"{label:24s} -> {e}")

# No images at all is not an error: it is the empty_request status.
print("no images                ->",
      (mp.run(config={"mediapipe": {"command": "hands"}}).config or {})["mediapipe"])

# reset closes the cached landmarkers; the next call rebuilds the one it needs.
print("reset                    ->", mediapipe("reset")[1])

## 8. Where this box fits

**MediaPipe wants one prominent, well-lit subject.** The landmarkers are built for
a person who fills a good share of the frame. On a surveillance frame — a wide
night-time street, people 30 px tall — it returns `counts = 0`, and that is the
model working as designed, not a failure of the box.

For that case the `yolo` box is the right tool: `command="pose"` gives 17 COCO
keypoints per person, tracked, and is built for small, multiple, partly occluded
people. The two are complementary — YOLO for *where the people are*, MediaPipe for
*the detailed skeleton* of one of them — and the way to combine them is to crop:

```python
yolo = Visionist("localhost:9068")
det = yolo.run(data={"images": [pathlib.Path(frame)]},
               config={"yolo": {"command": "detect", "parameters": {"classes": [0]}}})
# for each person box: crop it, upscale ~2-4x, send that crop to mediapipe,
# then add the crop's origin back to the returned pixel coordinates.
```

`PoseLandmarker` has no region-of-interest option (only the image embedder and the
interactive segmenter take one), so the crop has to happen on this side.

**Not in the box yet:** VIDEO / LIVE_STREAM running mode (needs monotonic
timestamps and the fleet's `session` convention, as `tapnext` does), and the other
MediaPipe tasks — holistic, segmentation, gesture recognition.

**From MATLAB or Octave**, the same box is `vsMediapipe`:

```matlab
out = vsMediapipe(cfg, 'images', "frame.jpg", 'command', "pose", ...
                  'model', "heavy", 'coordinates', "pixels");
xy  = squeeze(out.landmarks(1, 1:out.counts(1), :, 1:2));   % counts first, always
```